# Live Options Trading Analysis
## Covered Call Optimization with Real IBKR Data

This notebook connects to Interactive Brokers to fetch live option data and find optimal covered call strategies.

### Setup and Imports

In [ ]:
# Add core directory to path
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), 'core'))

# Import required libraries
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Import our custom modules
from YOUR_MAIN_INTERFACE import find_best_options, get_option_data

print("✅ Imports successful")
print(f"Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

### Configuration
Adjust these parameters for your trading strategy:

In [ ]:
# Trading Parameters
TICKER = 'AAPL'           # Stock symbol
EXPIRATION = '20250815'   # Option expiration (YYYYMMDD)
CAPITAL = 100000          # Capital to invest
MAX_CONTRACTS = 50        # Max contracts per strike
N_SIMULATIONS = 1000      # Monte Carlo simulations

print("Configuration:")
print(f"  Ticker: {TICKER}")
print(f"  Expiration: {EXPIRATION}")
print(f"  Capital: ${CAPITAL:,.2f}")
print(f"  Max Contracts/Strike: {MAX_CONTRACTS}")
print(f"  Simulations: {N_SIMULATIONS}")

### Step 1: Fetch Live Option Data
Connect to IB Gateway and fetch current option chain

In [ ]:
print("Fetching live option data...")
print("This may take 30-60 seconds...\n")

try:
    # Get option data with stock price
    bids, strikes, deltas, stock_price = get_option_data(
        TICKER, 
        EXPIRATION,
        return_stock_price=True
    )
    
    print(f"✅ Data fetched successfully!")
    print(f"\nCurrent {TICKER} Stock Price: ${stock_price:.2f}")
    print(f"Total Strikes Available: {len(strikes)}")
    print(f"Strike Range: ${strikes.min():.2f} - ${strikes.max():.2f}")
    
    # Create DataFrame for easier viewing
    option_data = pd.DataFrame({
        'Strike': strikes,
        'Bid': bids,
        'Delta': deltas
    })
    
except Exception as e:
    print(f"❌ Error fetching data: {e}")
    print("\nTroubleshooting:")
    print("1. Ensure IB Gateway is running on port 8000")
    print("2. Check market hours (9:30 AM - 4:00 PM ET)")
    print("3. Verify OPRA subscription is active")

### Step 2: View Option Chain
Examine strikes near the money

In [ ]:
# Find ATM strikes
atm_idx = (option_data['Strike'] - stock_price).abs().argmin()
start_idx = max(0, atm_idx - 5)
end_idx = min(len(option_data), atm_idx + 6)

# Display near-ATM options
print("Near-ATM Call Options:")
print("=" * 50)

atm_data = option_data.iloc[start_idx:end_idx].copy()
atm_data['Moneyness'] = atm_data['Strike'].apply(
    lambda x: 'ATM' if abs(x - stock_price) < 1 else ('ITM' if x < stock_price else 'OTM')
)
atm_data['Premium/Contract'] = atm_data['Bid'] * 100

pd.set_option('display.float_format', '{:.2f}'.format)
display(atm_data[['Strike', 'Moneyness', 'Bid', 'Delta', 'Premium/Contract']])

### Step 3: Find Optimal Covered Call Strategy
Run optimization to find best strikes to sell

In [ ]:
print(f"Running covered call optimization with ${CAPITAL:,.0f} capital...")
print("=" * 60)

try:
    # Run optimization
    results = find_best_options(
        ticker=TICKER,
        expiration=EXPIRATION,
        capital=CAPITAL,
        max_contracts_per_strike=MAX_CONTRACTS,
        n_simulations=N_SIMULATIONS
    )
    
    # Display results
    print("\n📊 OPTIMIZATION RESULTS:")
    print("=" * 60)
    
    print(f"\n💰 Capital Allocation:")
    print(f"  Shares to Buy: {results['shares_needed']:,}")
    print(f"  Share Cost: ${results['share_cost']:,.2f}")
    print(f"  Premium Collected: ${results['premium_collected']:,.2f}")
    print(f"  Net Investment: ${results['share_cost'] - results['premium_collected']:,.2f}")
    
    print(f"\n📈 Expected Performance:")
    print(f"  Expected P&L: ${results['expected_pnl']:,.2f}")
    print(f"  Win Rate: {results['win_rate']:.1%}")
    print(f"  Return on Capital: {results['return_pct']:.2%}")
    print(f"  Max Gain: ${results['max_gain']:,.2f}")
    print(f"  95% VaR: ${results['var_95']:,.2f}")
    
    print(f"\n🎯 Optimal Positions:")
    for strike, qty in results['optimal_positions'].items():
        idx = option_data[option_data['Strike'] == strike].index[0]
        bid = option_data.loc[idx, 'Bid']
        delta = option_data.loc[idx, 'Delta']
        print(f"  • Sell {qty} calls at ${strike:.2f} strike")
        print(f"    (Bid: ${bid:.2f}, Delta: {delta:.3f}, Premium: ${bid * qty * 100:.2f})")
    
except Exception as e:
    print(f"❌ Optimization failed: {e}")

### Step 4: Visualize Risk/Return Profile

In [ ]:
import matplotlib.pyplot as plt

if 'results' in locals():
    # Create P&L chart at different stock prices
    test_prices = np.linspace(stock_price * 0.8, stock_price * 1.2, 50)
    pnl_values = []
    
    for price in test_prices:
        # Calculate P&L at each price
        shares = results['shares_needed']
        stock_pnl = (price - stock_price) * shares
        
        # Add option P&L
        option_pnl = results['premium_collected']
        for strike, qty in results['optimal_positions'].items():
            if price > strike:
                # Shares called away at strike
                option_pnl += (strike - stock_price) * qty * 100
            else:
                # Keep shares, already counted in stock_pnl
                pass
        
        total_pnl = stock_pnl + option_pnl if price <= max(results['optimal_positions'].keys()) else \
                   (max(results['optimal_positions'].keys()) - stock_price) * shares + results['premium_collected']
        pnl_values.append(total_pnl)
    
    # Plot
    plt.figure(figsize=(12, 6))
    plt.plot(test_prices, pnl_values, 'b-', linewidth=2, label='Covered Call P&L')
    plt.axhline(y=0, color='k', linestyle='-', alpha=0.3)
    plt.axvline(x=stock_price, color='r', linestyle='--', alpha=0.5, label=f'Current Price (${stock_price:.2f})')
    
    # Mark strike prices
    for strike in results['optimal_positions'].keys():
        plt.axvline(x=strike, color='g', linestyle='--', alpha=0.3)
        plt.text(strike, max(pnl_values)*0.9, f'${strike:.0f}', rotation=45, ha='right')
    
    plt.xlabel('Stock Price at Expiration', fontsize=12)
    plt.ylabel('Profit/Loss ($)', fontsize=12)
    plt.title(f'Covered Call P&L Profile - {TICKER} {EXPIRATION}', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.legend()
    
    # Add shaded regions
    plt.fill_between(test_prices, 0, pnl_values, where=(np.array(pnl_values) > 0), 
                     alpha=0.3, color='green', label='Profit Zone')
    plt.fill_between(test_prices, 0, pnl_values, where=(np.array(pnl_values) < 0), 
                     alpha=0.3, color='red', label='Loss Zone')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nKey Metrics:")
    print(f"  Breakeven: ${stock_price - results['premium_collected']/results['shares_needed']:.2f}")
    print(f"  Max Profit at: ${max(results['optimal_positions'].keys()):.2f}")
    print(f"  Protected until: ${stock_price - results['premium_collected']/results['shares_needed']:.2f} (-{(results['premium_collected']/results['share_cost'])*100:.1f}%)")

### Step 5: Generate Trade Execution Summary

In [ ]:
if 'results' in locals():
    print("=" * 60)
    print("📋 TRADE EXECUTION CHECKLIST")
    print("=" * 60)
    
    print(f"\n1️⃣ BUY SHARES:")
    print(f"   Order Type: Market/Limit Buy")
    print(f"   Symbol: {TICKER}")
    print(f"   Quantity: {results['shares_needed']} shares")
    print(f"   Estimated Cost: ${results['share_cost']:,.2f}")
    
    print(f"\n2️⃣ SELL CALL OPTIONS (after shares are filled):")
    for i, (strike, qty) in enumerate(results['optimal_positions'].items(), 1):
        idx = option_data[option_data['Strike'] == strike].index[0]
        bid = option_data.loc[idx, 'Bid']
        
        print(f"\n   Order #{i}:")
        print(f"   Action: SELL TO OPEN")
        print(f"   Symbol: {TICKER}")
        print(f"   Expiration: {EXPIRATION}")
        print(f"   Strike: ${strike:.2f}")
        print(f"   Type: CALL")
        print(f"   Quantity: {qty} contracts")
        print(f"   Limit Price: ${bid:.2f} or better")
        print(f"   Expected Premium: ${bid * qty * 100:.2f}")
    
    print(f"\n3️⃣ VERIFY POSITIONS:")
    print(f"   ✓ Long {results['shares_needed']} shares {TICKER}")
    print(f"   ✓ Short {sum(results['optimal_positions'].values())} call contracts")
    print(f"   ✓ All shares are covered (no naked calls)")
    
    print(f"\n⚠️ RISK REMINDER:")
    print(f"   • Maximum Gain: ${results['max_gain']:,.2f}")
    print(f"   • If called away: Shares sold at strike price(s)")
    print(f"   • If not called: Keep shares + premium")
    print(f"   • Assignment can happen before expiration if ITM")

### Step 6: Export Results

In [ ]:
# Save results to CSV for record keeping
if 'results' in locals():
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"covered_call_{TICKER}_{EXPIRATION}_{timestamp}.csv"
    
    # Create summary DataFrame
    summary_data = {
        'Timestamp': [datetime.now()],
        'Ticker': [TICKER],
        'Expiration': [EXPIRATION],
        'Stock_Price': [stock_price],
        'Capital': [CAPITAL],
        'Shares': [results['shares_needed']],
        'Premium_Collected': [results['premium_collected']],
        'Expected_PnL': [results['expected_pnl']],
        'Win_Rate': [results['win_rate']],
        'Return_Pct': [results['return_pct']],
        'Strikes': [str(results['optimal_positions'])]
    }
    
    summary_df = pd.DataFrame(summary_data)
    summary_df.to_csv(filename, index=False)
    
    print(f"✅ Results saved to: {filename}")
    
    # Display summary
    print("\nSummary Table:")
    display(summary_df.drop('Timestamp', axis=1))

### Additional Analysis Tools

In [ ]:
# Compare different capital amounts
capital_levels = [50000, 75000, 100000, 150000, 200000]
comparison_results = []

print("Analyzing different capital levels...\n")

for capital in capital_levels:
    try:
        result = find_best_options(
            ticker=TICKER,
            expiration=EXPIRATION,
            capital=capital,
            n_simulations=200  # Fewer simulations for speed
        )
        
        comparison_results.append({
            'Capital': capital,
            'Shares': result['shares_needed'],
            'Contracts': sum(result['optimal_positions'].values()),
            'Premium': result['premium_collected'],
            'Expected_PnL': result['expected_pnl'],
            'Return_%': result['return_pct'] * 100
        })
    except:
        pass

if comparison_results:
    comparison_df = pd.DataFrame(comparison_results)
    print("Capital Comparison Analysis:")
    display(comparison_df)
    
    # Plot returns
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.plot(comparison_df['Capital'], comparison_df['Return_%'], 'bo-')
    plt.xlabel('Capital ($)')
    plt.ylabel('Return (%)')
    plt.title('Return vs Capital')
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.plot(comparison_df['Capital'], comparison_df['Expected_PnL'], 'go-')
    plt.xlabel('Capital ($)')
    plt.ylabel('Expected P&L ($)')
    plt.title('Expected P&L vs Capital')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

### Quick Functions for Future Use

In [ ]:
def quick_analysis(ticker, expiration, capital):
    """Quick covered call analysis for any ticker"""
    try:
        results = find_best_options(
            ticker=ticker,
            expiration=expiration,
            capital=capital,
            n_simulations=500
        )
        
        print(f"\n{ticker} Covered Call Analysis:")
        print("=" * 40)
        print(f"Capital: ${capital:,.0f}")
        print(f"Shares: {results['shares_needed']}")
        print(f"Premium: ${results['premium_collected']:,.2f}")
        print(f"Expected P&L: ${results['expected_pnl']:,.2f}")
        print(f"Return: {results['return_pct']:.2%}")
        print(f"Win Rate: {results['win_rate']:.1%}")
        print(f"\nStrikes to sell: {results['optimal_positions']}")
        
        return results
    except Exception as e:
        print(f"Error: {e}")
        return None

# Example usage:
# quick_analysis('TSLA', '20250815', 100000)
# quick_analysis('SPY', '20250815', 200000)

---
### Notes
- Always ensure IB Gateway is running before executing
- Market hours: 9:30 AM - 4:00 PM ET
- Requires OPRA subscription for live options data
- This notebook uses real money - trade carefully!